In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))


### ablation experiment

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path


def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def retrive_mean(results, num, runstep):
	results = results.reshape(num+1, runstep)
	return results.mean(axis=1)    

def retrive_H_star(path):
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
    
    

def compare_ablation(paths):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
#     star drop1 drop2 drop12 
	colors = ['#7FBF7B', '#333333', '#0070C0', '#DCA5C3']
# 500 500 4 3
# 	interval = [0, 10, 11, 10]
# 1000 1000 4 3
# 	interval = [0, 7, 5, 3]
	interval = [0, 14, 13, 12]
	markers = ['s', 's', 's', 's']
	linestyles = ['-', ':', ':', (0, (1, 1))]
	marker_size = [4, 4, 4, 4]
	marker_width = 1    
	marker_density = 10
    
	for k in range(3):
		with open(paths[3-k], 'r', encoding='utf-8') as f:
			data = json.load(f)
			results = np.array(data['results'])
			stop, num = data['p_interval'][0], data['p_interval'][1]
			runstep = data['runstep']
			type_ana = data['type_ana']
		ps = generate_interval((stop, num))         
		means = retrive_mean(results, num, runstep)    
# 		plt.plot(ps, means, color=colors[3-k], lw=7, linestyle=linestyles[3-k])
		ax.scatter(ps[::interval[3-k]], means[::interval[3-k]], color=colors[3-k], marker='s', s=80, edgecolors='white', linewidths=0.8)
	k=3
	with open(paths[3-k], 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		stop, num = data['p_interval'][0], data['p_interval'][1]
		runstep = data['runstep']
		type_ana = data['type_ana']
	ps = generate_interval((stop, num))         
	means = retrive_mean(results, num, runstep)    
	plt.plot(ps, means, color=colors[3-k], lw=3, linestyle=linestyles[3-k])
    
	if type_ana == 'ana_12':
		ax.set_ylim(0.95, 2.05)
		ax.set_yticks([1.0, 1.5, 2.0])
		ax.set_xticks([0.0, 0.2, 0.4])
        
	if type_ana == 'ana_11':
		ax.set_ylim(-3, 67)
		ax.set_yticks([0, 32, 64])
        
# 	if type_ana == 'ana_2':
# 		ax.set_ylim(-6, 206)
# 		ax.set_yticks([0, 100, 200])

# 	ax.tick_params(axis='y', labelleft=False)    
	ax.set_xticks([0.0, 0.2, 0.4])
	ax.tick_params(axis='both', labelsize=18)


    
	filename = f'{type_ana}' 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
	save_dir_figs=f"figs_ablation/"
	os.makedirs(save_dir_figs, exist_ok=True)
	output_path = os.path.join(save_dir_figs, filename)
	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
#  100 100 4 3 
# compare_ablation(['results_compare_er/ana_11/200_1000t17_333737.json',
#                  'results_ablation_c/drop1/ana_11t32_902124.json',
#                  'results_ablation_c/drop2/ana_11t29_537110.json',
#                  'results_ablation_c/drop12/ana_11t09_990920.json'])

# compare_ablation(['results_compare_er/ana_12/200_1000t25_718391.json',
#                  'results_ablation_c/drop1/ana_12t46_510258.json',
#                  'results_ablation_c/drop2/ana_12t29_060706.json',
#                  'results_ablation_c/drop12/ana_12t59_086252.json'])

# compare_ablation(['results_compare_er/ana_2/200_500t17_376323.json',
#                  'results_ablation_c/drop1/ana_2t12_452029.json',
#                  'results_ablation_c/drop2/ana_2t58_195509.json',
#                  'results_ablation_c/drop12/ana_2t41_216358.json'])

#  100 100 6 3 

# compare_ablation(['results_compare_erba/ana_11/200_1000t06_851028.json',
#                  'results_ablation_c/drop1/ana_11t24_778650.json',
#                  'results_ablation_c/drop2/ana_11t58_443380.json',
#                  'results_ablation_c/drop12/ana_11t31_064959.json'])

# compare_ablation(['results_compare_erba/ana_12/200_1000t16_495703.json',
#                  'results_ablation_c/drop1/ana_12t35_280677.json',
#                  'results_ablation_c/drop2/ana_12t47_195631.json',
#                  'results_ablation_c/drop12/ana_12t20_597611.json'])

# compare_ablation(['results_compare_erba/ana_2/200_1000t01_311066.json',
#                  'results_ablation_c/drop1/ana_2t02_145812.json',
#                  'results_ablation_c/drop2/ana_2t09_767653.json',
#                  'results_ablation_c/drop12/ana_2t42_089557.json'])

#  500 500 4 3 
# compare_ablation(['results_ablation_c/star/start57_744794.json',
#                  'results_ablation_c/drop1/ana_11t05_382982.json',
#                  'results_ablation_c/drop2/ana_11t51_459947.json',
#                  'results_ablation_c/drop12/ana_11t07_050588.json'])

# compare_ablation(['results_ablation_c/star/start20_850330.json',
#                  'results_ablation_c/drop1/drop1t54_617898.json',
#                  'results_ablation_c/drop2/drop2t14_473928.json',
#                  'results_ablation_c/drop12/drop12t41_898931.json'])



# #  500 500 6 3 
# compare_ablation(['results_compare_sa/ana_11/200_1000t02_743989.json',
#                  'results_ablation_c/drop1/ana_11t33_538627.json',
#                  'results_ablation_c/drop2/ana_11t13_069939.json',
#                  'results_ablation_c/drop12/ana_11t45_527504.json'])

# compare_ablation(['results_compare_sa/ana_12/200_1000t58_261484.json',
#                  'results_ablation_c/drop1/ana_12t45_545853.json',
#                  'results_ablation_c/drop2/ana_12t31_261920.json',
#                  'results_ablation_c/drop12/ana_12t10_094419.json'])

# compare_ablation(['results_ablation_c/star/start52_862476.json',
#                  'results_ablation_c/drop1/ana_2t23_114442.json',
#                  'results_ablation_c/drop2/ana_2t48_317336.json',
#                  'results_ablation_c/drop12/ana_2t32_286389.json'])

#  1000 1000 4 3 
# compare_ablation(['results_ablation_c/star/start35_616946.json',
#                  'results_ablation_c/drop1/drop1t35_684764.json',
#                  'results_ablation_c/drop2/drop2t56_764961.json',
#                  'results_ablation_c/drop12/drop12t23_255865.json'])

# compare_ablation(['results_ablation_c/star/start48_783387.json',
#                  'results_ablation_c/drop1/drop1t37_797865.json',
#                  'results_ablation_c/drop2/drop2t00_310909.json',
#                  'results_ablation_c/drop12/drop12t36_666622.json'])

# #  1000 1000 6 3 
# compare_ablation(['results_ablation_c/star/start19_989608.json',
#                  'results_ablation_c/drop1/drop1t05_459939.json',
#                  'results_ablation_c/drop2/drop2t04_672706.json',
#                  'results_ablation_c/drop12/drop12t25_035511.json'])

compare_ablation(['results_ablation_c/star/start28_183905.json',
                 'results_ablation_c/drop1/drop1t21_660813.json',
                 'results_ablation_c/drop2/drop2t42_419782.json',
                 'results_ablation_c/drop12/drop12t08_035056.json'])



### sensitivity analysis-R

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path


def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def retrive_mean(results, num, runstep):
	results = results.reshape(num+1, runstep)
	return results.mean(axis=1)    

def retrive_H_star(path):
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
    
    

def compare_sensiR(paths):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
#     star drop1 drop2 drop12 
	colors = ['#0070C0', '#7FBF7B', '#333333']
# 500 500 4 3
# 	interval = [0, 10, 11, 10]
# 1000 1000 4 3
# 	interval = [0, 7, 5, 3]
	interval = [0, 14, 13, 12]
	markers = ['^', '^', '^']
	linestyles = ['--', '-', '-.']
	marker_size = [4, 4, 4, 4]
	marker_width = 1    
	marker_density = 10
    
	for k in range(3):
		with open(paths[k], 'r', encoding='utf-8') as f:
			data = json.load(f)
			results = np.array(data['results'])
			stop, num = data['p_interval'][0], data['p_interval'][1]
			runstep = data['runstep']
			type_ana = data['type_ana']
		ps = generate_interval((stop, num))         
		means = retrive_mean(results, num, runstep)
		if k != 1:
			ax.scatter(ps[::11], means[::11], color=colors[k], 
                       marker=markers[k], s=100, edgecolors='white', linewidths=0.8,
                      alpha=0.9)
		if k == 1:
			plt.plot(ps, means, color=colors[k], lw=2, linestyle=linestyles[k])

    
	if type_ana == 'ana_12':
		ax.set_ylim(0.95, 2.05)
		ax.set_yticks([1.0, 1.5, 2.0])
		ax.set_xticks([0.0, 0.2, 0.4])
        
	if type_ana == 'ana_11':
		ax.set_ylim(-3, 67)
		ax.set_yticks([0, 32, 64])
        
	if type_ana == 'ana_2':
		ax.set_ylim(-6, 206)
		ax.set_yticks([0, 100, 200])

# 	ax.tick_params(axis='y', labelleft=False)    
	ax.set_xticks([0.0, 0.2, 0.4])
	ax.tick_params(axis='both', labelsize=18)


    
	filename = f'{type_ana}' 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
	save_dir_figs=f"figs_sensiR/"
	os.makedirs(save_dir_figs, exist_ok=True)
	output_path = os.path.join(save_dir_figs, filename)
	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
# R 2 3 4 5 
# compare_sensiR(['results_sensir/ana_11/2t57_442275.json',
#                  'results_sensir/ana_11/3t58_791832.json',
#                  'results_sensir/ana_11/4t00_223995.json'])

# compare_sensiR(['results_sensir/ana_12/2t51_431144.json',
#                  'results_sensir/ana_12/3t47_080932.json',
#                  'results_sensir/ana_12/4t56_645861.json'])

# compare_sensiR(['results_sensir/ana_2/2t16_704859.json',
#                  'results_sensir/ana_2/3t40_505331.json',
#                  'results_sensir/ana_2/4t38_598566.json'])

# N/T star 0.5 1.0 2.0
compare_sensiR(['results_sensiNT/ana_11/0_5t24_337145.json',
                 'results_sensiNT/ana_11/1_0t25_643324.json',
                 'results_sensiNT/ana_11/2_0t26_468323.json'])

compare_sensiR(['results_sensiNT/ana_12/0_5t44_701708.json',
                 'results_sensiNT/ana_12/1_0t38_865943.json',
                 'results_sensiNT/ana_12/2_0t54_095048.json'])

compare_sensiR(['results_sensiNT/ana_2/0_5t22_667321.json',
                 'results_sensiNT/ana_2/1_0t50_789382.json',
                 'results_sensiNT/ana_2/2_0t19_454721.json'])

In [ ]:
import random
import math
import time
import numpy as np
import copy
import warnings
warnings.simplefilter('ignore')
import json
import os
from pathlib import Path


def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def retrive_mean(results, num, runstep):
	results = results.reshape(num+1, runstep)
	return results.mean(axis=1)    

def retrive_H_star(path):
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
    
def map2043(results):
    max_value = max(results)
    results_01 = [v / max_value for v in results]
    return [43 * v for v in results_01]

def obtain_means(paths):
	results_raw = []
	for p in paths:
		with open(p, 'r', encoding='utf-8') as f:
			data = json.load(f)
			results = np.array(data['results'])
			stop, num = data['p_interval'][0], data['p_interval'][1]
			runstep = data['runstep']     
		means = retrive_mean(results, num, runstep)    
		avg = sum(means) / len(means)
		results_raw.append(avg)
	mapped = map2043(results_raw)
	return mapped

def obtain_means(path_sa_star, path_mu2, path_gr, path_erin, path_ba):
	results_raw = []
	with open(path_sa_star, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		results_sa = np.array(data['results_sa'])
		stop, num = data['p_interval']
		runstep = data['runstep']

	with open(path_mu2, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results_mu2 = np.array(data['results_sa'])
        
	with open(path_gr, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results_gr = np.array(data['results_gr'])  
        
	means_star = retrive_mean(results, num, runstep)
	means_sa = retrive_mean(results_sa, num, runstep)   
	means_gr = retrive_mean(results_gr, num, runstep)
	means_mu2 = retrive_mean(results_mu2, num, runstep)
    
	with open(path_ba, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results_ba = np.array(data['results_ba'])

	means_ba = retrive_mean(results_ba, num, runstep)
    
	with open(path_erin, 'r', encoding='utf-8') as f:
		data = json.load(f)
		stop, num = data['p_interval']
		runstep = data['runstep']
		results_er = np.array(data['results_er'])
        
	means_er = retrive_mean(results_er, num, runstep)
    
	avg_star = sum(means_star) / len(means_star)
	avg_sa = sum(means_sa) / len(means_sa)
	avg_gr = sum(means_gr) / len(means_gr)    
	avg_mu2 = sum(means_mu2) / len(means_mu2)
	avg_ba = sum(means_ba) / len(means_ba)
	avg_er = sum(means_er) / len(means_er)
    
	raw = [avg_star, avg_sa, avg_gr, avg_mu2, avg_ba, avg_er]
	mapped = map2043(raw)
	return mapped

In [ ]:
# 100 100 3 2 star sa gr mu2 ba er
# ana_11
# paths = ['results_sensir/ana_11/2t57_442275.json',
#     'results_sensir_all/ana_11/sat17_170541.json',
# 'results_sensir_all/ana_11/grt18_381948.json',
# 'results_sensir_all/ana_11/mu2t19_634981.json',
# 'results_sensir_all/ana_11/bat20_810946.json',
# 'results_sensir_all/ana_11/ert21_973658.json']
# obtain_means(paths)

# # ana_12
# paths = ['results_sensir/ana_12/2t51_431144.json',
# 'results_sensir_all/ana_12/sat54_185233.json',
# 'results_sensir_all/ana_12/grt03_002919.json',
# 'results_sensir_all/ana_12/mu2t12_590912.json',
# 'results_sensir_all/ana_12/bat21_095593.json',
# 'results_sensir_all/ana_12/ert29_511410.json']
# obtain_means(paths)

# # ana_2
# paths = ['results_sensir/ana_2/2t16_704859.json',
# 'results_sensir_all/ana_2/sat49_218425.json',
# 'results_sensir_all/ana_2/grt04_017561.json',
# 'results_sensir_all/ana_2/mu2t23_676226.json',
# 'results_sensir_all/ana_2/bat34_541412.json',
# 'results_sensir_all/ana_2/ert45_417177.json']
# obtain_means(paths)

# 100 100 4 3 star sa gr mu2 ba er
# obtain_means('results_compare_sa/ana_11/200_1000t26_919477.json', 
#              'results_single_mu2/ana_11/200_1000t03_052832.json', 
#              'results_single_greedy/ana_11/200_1000t02_980236.json', 
#              'results_compare_er/ana_11/200_1000t17_333737.json', 
#              'results_compare_ba/ana_11/200_1000t23_598234.json')

# obtain_means('results_compare_sa/ana_12/200_1000t05_530290.json', 
#              'results_single_mu2/ana_12/200_1000t09_144374.json', 
#              'results_single_greedy/ana_12/200_1000t50_579754.json', 
#              'results_compare_er/ana_12/200_1000t25_718391.json', 
#              'results_compare_ba/ana_12/200_1000t03_564217.json')

# obtain_means('results_compare_sa/ana_2/200_1000t01_483425.json', 
#              'results_single_mu2/ana_2/200_1000t36_060675.json', 
#              'results_single_greedy/ana_2/200_1000t41_443874.json', 
#              'results_compare_er/ana_2/200_500t17_376323.json', 
#              'results_compare_ba/ana_2/200_1000t15_849751.json')

# 100 100 5 4 star sa gr mu2 ba er
# # ana_11
# paths = ['results_sensir/ana_11/4t00_223995.json',
#     'results_sensir_all/ana_11/sat23_448397.json',
# 'results_sensir_all/ana_11/grt24_800045.json',
# 'results_sensir_all/ana_11/mu2t26_209411.json',
# 'results_sensir_all/ana_11/bat27_535597.json',
# 'results_sensir_all/ana_11/ert28_853183.json']
# obtain_means(paths)

# # # ana_12
# paths = ['results_sensir/ana_12/4t56_645861.json',
# 'results_sensir_all/ana_12/sat58_453824.json',
# 'results_sensir_all/ana_12/grt10_579368.json',
# 'results_sensir_all/ana_12/mu2t23_508023.json',
# 'results_sensir_all/ana_12/bat37_971933.json',
# 'results_sensir_all/ana_12/ert49_159851.json']
# obtain_means(paths)

# # # ana_2
# paths = ['results_sensir/ana_2/4t08_161205.json',
# 'results_sensir_all/ana_2/sat20_927437.json',
# 'results_sensir_all/ana_2/grt49_658057.json',
# 'results_sensir_all/ana_2/mu2t21_495244.json',
# 'results_sensir_all/ana_2/bat38_508537.json',
# 'results_sensir_all/ana_2/ert02_160066.json']
# obtain_means(paths)

### sensitivity analysis-NT

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path


def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def retrive_mean(results, num, runstep):
	results = results.reshape(num+1, runstep)
	return results.mean(axis=1)    

def retrive_H_star(path):
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
    
    

def compare_sensiNT(paths):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
#     star drop1 drop2 drop12 
	colors = ['#7FBF7B', '#333333', '#0070C0', '#DCA5C3']
# 500 500 4 3
# 	interval = [0, 10, 11, 10]
# 1000 1000 4 3
# 	interval = [0, 7, 5, 3]
	interval = [0, 14, 13, 12]
	markers = ['s', 's', 's', 's']
	linestyles = ['-', ':', ':', (0, (1, 1))]
	marker_size = [4, 4, 4, 4]
	marker_width = 1    
	marker_density = 10
    
	for k in range(3):
		with open(paths[2-k], 'r', encoding='utf-8') as f:
			data = json.load(f)
			results = np.array(data['results'])
			stop, num = data['p_interval'][0], data['p_interval'][1]
			runstep = data['runstep']
			type_ana = data['type_ana']
		ps = generate_interval((stop, num))         
		means = retrive_mean(results, num, runstep)    
		plt.plot(ps, means, color=colors[2-k], lw=7, linestyle=linestyles[2-k])
# 		ax.scatter(ps, means, color=colors[2-k], marker='s', s=80, edgecolors='white', linewidths=0.8)
    
	if type_ana == 'ana_12':
		ax.set_ylim(0.95, 2.05)
		ax.set_yticks([1.0, 1.5, 2.0])
		ax.set_xticks([0.0, 0.2, 0.4])
        
	if type_ana == 'ana_11':
		ax.set_ylim(-3, 67)
		ax.set_yticks([0, 32, 64])
        
# 	if type_ana == 'ana_2':
# 		ax.set_ylim(-6, 206)
# 		ax.set_yticks([0, 100, 200])

# 	ax.tick_params(axis='y', labelleft=False)    
	ax.set_xticks([0.0, 0.2, 0.4])
	ax.tick_params(axis='both', labelsize=18)


    
# 	filename = f'{type_ana}' 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
# 	save_dir_figs=f"figs_sensiNT/"
# 	os.makedirs(save_dir_figs, exist_ok=True)
# 	output_path = os.path.join(save_dir_figs, filename)
# 	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
# N/T star 0.5 1.0 2.0
compare_sensiNT(['results_sensiNT/ana_11/0_5t24_337145.json',
                 'results_sensiNT/ana_11/1_0t25_643324.json',
                 'results_sensiNT/ana_11/2_0t26_468323.json'])

compare_sensiNT(['results_sensiNT/ana_12/0_5t44_701708.json',
                 'results_sensiNT/ana_12/1_0t38_865943.json',
                 'results_sensiNT/ana_12/2_0t54_095048.json'])

compare_sensiNT(['results_sensiNT/ana_2/0_5t22_667321.json',
                 'results_sensiNT/ana_2/1_0t50_789382.json',
                 'results_sensiNT/ana_2/2_0t19_454721.json'])